In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# For saving plots as PDF
from matplotlib.backends.backend_pdf import PdfPages

# For model building and evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score, roc_curve, auc

# Load the dataset from local CSV file
file_path = 'D:/22BT30025-P2-ShuvamVidyarthy/framingham.csv'
df = pd.read_csv(file_path)

# Display basic information
print("Dataset Info:")
print(df.info())
print("\nFirst 5 rows:")
print(df.head())


Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4240 entries, 0 to 4239
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   male             4240 non-null   int64  
 1   age              4240 non-null   int64  
 2   education        4135 non-null   float64
 3   currentSmoker    4240 non-null   int64  
 4   cigsPerDay       4211 non-null   float64
 5   BPMeds           4187 non-null   float64
 6   prevalentStroke  4240 non-null   int64  
 7   prevalentHyp     4240 non-null   int64  
 8   diabetes         4240 non-null   int64  
 9   totChol          4190 non-null   float64
 10  sysBP            4240 non-null   float64
 11  diaBP            4240 non-null   float64
 12  BMI              4221 non-null   float64
 13  heartRate        4239 non-null   float64
 14  glucose          3852 non-null   float64
 15  TenYearCHD       4240 non-null   int64  
dtypes: float64(9), int64(7)
memory usage: 530.1 KB

In [3]:
# Check for missing values
print("Missing Values Before Imputation:")
print(df.isnull().sum())

# Impute numerical columns with median
numerical_cols = ['cigsPerDay', 'BPMeds', 'totChol', 'BMI', 'heartRate', 'glucose']
for col in numerical_cols:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    print(f"Filled missing values in {col} with median value {median_value}")

# Impute 'education' with mode
mode_education = df['education'].mode()[0]
df['education'] = df['education'].fillna(mode_education)
print(f"Filled missing values in education with mode value {mode_education}")

# Verify that all missing values are handled
print("\nMissing Values After Imputation:")
print(df.isnull().sum())

Missing Values Before Imputation:
male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totChol            0
sysBP              0
diaBP              0
BMI                0
heartRate          0
glucose            0
TenYearCHD         0
dtype: int64
Filled missing values in cigsPerDay with median value 0.0
Filled missing values in BPMeds with median value 0.0
Filled missing values in totChol with median value 234.0
Filled missing values in BMI with median value 25.4
Filled missing values in heartRate with median value 75.0
Filled missing values in glucose with median value 78.0
Filled missing values in education with mode value 1.0

Missing Values After Imputation:
male               0
age                0
education          0
currentSmoker      0
cigsPerDay         0
BPMeds             0
prevalentStroke    0
prevalentHyp       0
diabetes           0
totC

In [5]:
# Create a PDF file to save all plots
pdf_path = 'eda_report.pdf'
pdf = PdfPages(pdf_path)

sns.set(style="whitegrid")

# 1. Distribution of the target variable
plt.figure(figsize=(6,4))
sns.countplot(x='TenYearCHD', data=df, color='skyblue')  # Use color instead of palette
plt.title('Distribution of 10 Year CHD Risk')
plt.xlabel('10 Year CHD Risk')
plt.ylabel('Count')
pdf.savefig()
plt.close()

# 2. Correlation heatmap
plt.figure(figsize=(12,10))
corr = df.corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Heatmap')
pdf.savefig()
plt.close()

# 3. Pairplot for selected features
selected_features = ['age', 'sysBP', 'BMI', 'glucose', 'TenYearCHD']
sns.pairplot(df[selected_features], hue='TenYearCHD', palette='coolwarm', diag_kind='kde')
plt.suptitle('Pair Plot of Selected Features', y=1.02)
pdf.savefig()
plt.close()

# 4. Age distribution by CHD status
plt.figure(figsize=(8,6))
sns.kdeplot(data=df, x='age', hue='TenYearCHD', fill=True, palette='muted')
plt.title('Age Distribution by CHD Status')
pdf.savefig()
plt.close()

# 5. Cholesterol distribution by CHD status
plt.figure(figsize=(8,6))
sns.boxplot(x='TenYearCHD', y='totChol', data=df, hue='TenYearCHD', palette='pastel')  # Add hue
plt.title('Total Cholesterol by CHD Status')
pdf.savefig()
plt.close()

# 6. Scatter plot of BMI vs Glucose colored by CHD status
plt.figure(figsize=(8,6))
sns.scatterplot(x='BMI', y='glucose', hue='TenYearCHD', data=df, palette='coolwarm')
plt.title('BMI vs Glucose by CHD Status')
pdf.savefig()
plt.close()

# Close the PDF file
pdf.close()

print(f"EDA plots saved in {pdf_path}")

EDA plots saved in eda_report.pdf


In [6]:
# Split features and target
X = df.drop(columns=['TenYearCHD'])
y = df['TenYearCHD']

# Standard scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

print("Data scaling and splitting complete.")

Data scaling and splitting complete.


In [7]:
# Initialize logistic regression with hyperparameter tuning
param_grid = {'C': [0.01, 0.1, 1, 10, 100]}
grid = GridSearchCV(LogisticRegression(max_iter=1000, random_state=42), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)

# Best model
best_model = grid.best_estimator_
print("Best parameters:", grid.best_params_)

# Predict on test set
y_pred = best_model.predict(X_test)
y_probs = best_model.predict_proba(X_test)[:, 1]

# Evaluate
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_probs)
print(f"Test Accuracy: {accuracy:.4f}")
print(f"ROC AUC Score: {roc_auc:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Best parameters: {'C': 1}
Test Accuracy: 0.8550
ROC AUC Score: 0.7087

Confusion Matrix:
[[717   8]
 [115   8]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.99      0.92       725
           1       0.50      0.07      0.12       123

    accuracy                           0.85       848
   macro avg       0.68      0.53      0.52       848
weighted avg       0.81      0.85      0.80       848



In [ ]:
# Plot ROC curve
plt.figure(figsize=(8,6))
fpr, tpr, _ = roc_curve(y_test, y_probs)
roc_auc = auc(fpr, tpr)

plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='red', linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.show()
